# CoNLL-2003 NER — Load & Explore with `src` Packages

This notebook demonstrates how to use the project's `src` packages to:
1. Load an experiment config
2. Build the CoNLL-2003 dataloaders (with BPE-aware label alignment)
3. Inspect the dataset — label set, batch shape, example tokens


In [ ]:
import sys, os

# Make sure the workspace root is on the path so `src` is importable
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

print("Workspace root:", ROOT)


In [ ]:

# ── Environment check ─────────────────────────────────────────────────────────
import importlib.metadata, subprocess, sys

ds_version = importlib.metadata.version("datasets")
print(f"datasets version : {ds_version}")

major = int(ds_version.split(".")[0])
if major >= 3:
    print("⚠  datasets >= 3 does NOT support the conll2003 dataset script.")
    print("   Fix: pip install 'datasets<3'")
    print("   Running fix now...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "datasets<3", "-q"])
    print("✅  Downgrade complete – please restart the kernel, then re-run.")
else:
    print(f"✅  datasets {ds_version} is compatible with conll2003.")

# Quick load test
try:
    from datasets import load_dataset
    ds = load_dataset("conll2003", split="train[:1]")
    print(f"✅  conll2003 loaded OK – columns: {ds.column_names}")
except Exception as e:
    print(f"❌  conll2003 load failed: {e}")


## 1. Load the Experiment Config

The YAML lives at `src/configs/experiments/ner_conll2003_distilbert.yaml` and drives all hyperparameters.


In [ ]:
from src.configs import load_experiment_config

CONFIG_PATH = os.path.join(ROOT, "src/configs/experiments/ner_conll2003_distilbert.yaml")
cfg = load_experiment_config(CONFIG_PATH)

print(f"Experiment : {cfg.experiment_name}")
print(f"Dataset    : {cfg.dataset.name}  (max_len={cfg.dataset.max_length})")
print(f"Encoder    : {cfg.model.encoder_name}")
print(f"Batch size : {cfg.dataloader.batch_size}")
print(f"Smoothing  : mode={cfg.smoothing.mode}, sigma={cfg.smoothing.sigma}, k={cfg.smoothing.knn_k}")


## 2. Build the CoNLL-2003 Dataloaders

`build_conll_dataloaders` calls `datasets.load_dataset("conll2003")`, tokenises with DistilBERT's BPE tokeniser, and aligns NER labels to subword tokens (only the first subword of each word keeps its label; continuations get `-100`).


In [ ]:
from src.dataloaders import build_conll_dataloaders

data = build_conll_dataloaders(
    dataset_cfg=cfg.dataset,
    loader_cfg=cfg.dataloader,
    encoder_name=cfg.model.encoder_name,
)

print("=== Label set ===")
for idx, name in sorted(data.id2label.items()):
    print(f"  {idx:2d}  {name}")

print(f"\nTokenizer : {data.tokenizer_name}")
print(f"Train batches : {len(data.train_loader)}")
print(f"Val   batches : {len(data.val_loader)}")
print(f"Test  batches : {len(data.test_loader)}")


## 3. Inspect a Batch

Pull one batch from the training loader and print the tensor shapes plus a decoded example sentence with its NER tags.


In [ ]:
from transformers import AutoTokenizer

batch = next(iter(data.train_loader))

print("=== Batch tensor shapes ===")
for key, val in batch.items():
    print(f"  {key:20s}: {tuple(val.shape)}")

# Decode the first example in the batch
tokenizer = AutoTokenizer.from_pretrained(data.tokenizer_name)
input_ids = batch["input_ids"][0]
labels    = batch["labels"][0]
tokens    = tokenizer.convert_ids_to_tokens(input_ids)

print("\n=== First example: token → NER label ===")
print(f"{'Token':<20} Label")
print("-" * 32)
for tok, lbl_id in zip(tokens, labels.tolist()):
    lbl_name = data.id2label.get(lbl_id, "IGN") if lbl_id != -100 else "-"
    print(f"  {tok:<18} {lbl_name}")
